# micro-sam GPU segmentation in napari

Interactive + automatic segmentation with [micro-sam](https://github.com/computational-cell-analytics/micro-sam) using napari, running on the GPU.

**Kernel:** run this notebook on **`Python (micro_sam)`** (top-right kernel picker). That env has `micro_sam`, `napari`, and a CUDA build of PyTorch.

**Models** (`model_type`): use the microscopy-finetuned ones for best results —
`vit_b_lm` / `vit_l_lm` (light / fluorescence microscopy), `vit_b_em_organelles` (EM), or the vanilla SAM `vit_b` / `vit_l` / `vit_h`. The GPU is used automatically when available.

In [1]:
# Setup + GPU check
import torch
import napari
import micro_sam

print("micro_sam:", micro_sam.__version__)
print("napari   :", napari.__version__)
print("torch    :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device :", torch.cuda.get_device_name(0) if device == "cuda" else "cpu")

micro_sam: 1.8.4
napari   : 0.7.1
torch    : 2.12.0
CUDA available: True
Using device : NVIDIA GeForce RTX 3080


In [6]:
viewer = napari.Viewer()

: 

In [4]:
# Load a 2D image to segment (grayscale, RGB, or a single plane of a stack)
import tifffile

# TODO: point this at your image
image_path = r"C:\Users\isobe\The University of Manchester Dropbox\Isobel Taylor-Hearn\Bell+Marta\24wp\PP_NXD8_129_DIV24_24102025\Images\r01c02f42p01-ch1sk1fk1fl1.tiff"
image = tifffile.imread(image_path)

# If this is a multi-channel / z-stack, pick a single 2D plane, e.g.:
# image = image[z_index]           # a z-slice
# image = image[:, channel]        # a channel

print("image shape:", image.shape, "dtype:", image.dtype)

image shape: (2160, 2160) dtype: uint16


In [5]:
# Interactive annotation: launch the micro-sam 2D annotator on the GPU.
# Add point/box prompts in napari, then press "s" to segment the object and
# "Shift+S" for automatic (whole-image) segmentation. Embeddings are cached to
# `embedding_path` so re-opening the same image is fast.
from micro_sam.sam_annotator import annotator_2d

viewer = annotator_2d(
    image,
    model_type="vit_b_lm",
    embedding_path="embeddings/annotator_2d_vit_b_lm.zarr",
    device=device,
)
napari.run()

100%|###############################################| 375M/375M [00:00<?, ?B/s]
100%|#############################################| 38.4M/38.4M [00:00<?, ?B/s]
Compute Image Embeddings 2D: 100%|██████████| 1/1 [00:01<00:00,  1.65s/it]

PermissionError: [WinError 5] Access is denied: 'embeddings\\annotator_2d_vit_b_lm.zarr\\zarr.23295e77e1b240edb2e0834f53a8bcb2.partial' -> 'embeddings\\annotator_2d_vit_b_lm.zarr\\zarr.json'

In [ ]:
# Automatic segmentation (no prompts) -> instance label mask, then view in napari.
from micro_sam.automatic_segmentation import automatic_instance_segmentation, get_predictor_and_segmenter

predictor, segmenter = get_predictor_and_segmenter(
    model_type="vit_b_lm",
    device=device,
)

instances = automatic_instance_segmentation(
    predictor=predictor,
    segmenter=segmenter,
    input_path=image,
    ndim=2,
)
print("instances:", instances.shape, "num objects:", int(instances.max()))

viewer = napari.Viewer()
viewer.add_image(image, name="image")
viewer.add_labels(instances, name="micro-sam instances")
napari.run()